# C_setpoint_balance: constrained cross projections

New study, 16 November 2026: `0 <= P.0.1, P.1.0 < 0.1`. Previous learned examples and outputs were deleted. Fitness is unchanged; spike activity is diagnostic only. Run all cells with NumPy/Numba; SpikingNN is not needed. Until a new winner is exported, the default example is explicitly an **untrained generic controller**. Training remains off by default.

In [ ]:
from pathlib import Path
import sys
import json
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next(q for p in candidates for q in (p, p / "topubsrc")
            if (q / "model.py").is_file() and (q / "genome.py").is_file())
sys.path.insert(0, str(ROOT))
from genome import load_solution
from model import replay_genome, plot_replay, measure_activity, METRIC_COLUMNS, DT_MS
from evolution import profile_inputs, make_genome, seed_coordinates
EXPERIMENT = ROOT / "C_setpoint_balance"


## Recorded new search and three-bank results

After completion, these cells show the new validation-selected fast winner. Activity and test outcomes never select the winner. Until export, the table and learning curves are marked pending. The standalone README describes the planned sweep and initialization.

In [ ]:
RUN = None
catalog = ROOT / "published_results.json"
if catalog.exists():
    record = next(r for r in json.loads(catalog.read_text())
                  if r["experiment"] == EXPERIMENT.name[0] and r["domain"] == "fast")
    RUN = ROOT / Path(record["source"]).relative_to("topubsrc").parent
    settings = json.loads((RUN / "training.json").read_text())
    measured = json.loads((RUN / "result.json").read_text())
    print("Selected new run:", RUN.relative_to(ROOT))
    print("Search:", settings)
    lines = ["| Bank | Fitness | Strict success | Mean late RMSE (deg) |",
             "|---|---:|---:|---:|"]
    for bank in ("training", "validation", "test"):
        row = measured[bank]
        lines.append(f"| {bank} | {row['fitness']:.6f} | {row['success']}/{row['cases']} | {row['mean_late_rmse_deg']:.6f} |")
    display(Markdown("\n".join(lines)))
    activity_report = json.loads((RUN / "activity.json").read_text())
    print("Both neurons active in each late 1 s bin:", activity_report["both_active_every_late_bin_cases"],
          "/", activity_report["complete_late_windows"], "complete test episodes")
else:
    print("New study results are pending. No previous numerical results are loaded.")


## Recorded learning curves

C switches validation banks at its table-to-polynomial boundary; curves are shown separately across that boundary.

In [ ]:
if RUN is None:
    print("New learning curves are pending.")
else:
    import csv
    stages = [RUN / "table", RUN / "polynomial"] if (RUN / "table").exists() else [RUN / "training"]
    fig, ax = plt.subplots(figsize=(10, 3.5), layout="constrained")
    offset = 0
    for stage in stages:
        with (stage / "history.csv").open() as stream:
            history = list(csv.DictReader(stream))
        with (stage / "validation_history.csv").open() as stream:
            validation = list(csv.DictReader(stream))
        ax.plot([offset + int(r["generation"]) for r in history],
                [float(r["best_fitness"]) for r in history], ":", label=f"{stage.name}: best training")
        ax.step([offset + int(r["generation"]) for r in validation],
                [float(r["selected_fitness"]) for r in validation], where="post", label=f"{stage.name}: validation incumbent")
        if offset: ax.axvline(offset, color="grey", linestyle="--")
        offset += len(history)
    ax.set(xlabel="Evaluated generation (including both C stages)", ylabel="Fitness", ylim=(-.02,1.02))
    ax.legend(); ax.grid(alpha=.2)
    plt.show()


## Choose a new genome and trial

`genomes/fast.json` contains the committed constrained-controller snapshot; see `../GENOMES.md` for its provisional status and selection rule. If this file is absent, the default path triggers an explicitly labelled untrained generic controller. A custom missing path raises an error. Change the path to a completed new run's `best_genome.json` to examine it before the sweep finishes. Angles are degrees and speeds rad/s. A/B target must remain zero.

`DURATION_S` is the episode length; `FITNESS_WINDOW_S=None` uses the saved last-window setting (2 s). Use `RECORD_EVERY=1` for reliable spike diagnostics.

In [ ]:
GENOME_PATH = EXPERIMENT / "genomes" / "fast.json"
TARGET_DEG = 45.0
INITIAL_ANGLE_DEG = -30.0
INITIAL_SPEED_RAD_S = 0.0
DURATION_S = 10.0
FITNESS_WINDOW_S = None  # None: saved genome setting (2 s for article genomes).
RECORD_EVERY = 1  # Integration always uses 0.1 ms; 1 records every spike sample.

if GENOME_PATH.exists():
    genome, unit = load_solution(GENOME_PATH)
elif GENOME_PATH == EXPERIMENT / "genomes" / "fast.json":
    config, _ = profile_inputs(EXPERIMENT, "fast")
    genome = make_genome(config)
    unit = seed_coordinates(genome)
    print("UNTRAINED GENERIC CONTROLLER: new winner not exported yet.")
else:
    raise FileNotFoundError(GENOME_PATH)
physical, command = genome.decode(unit)
assert np.all(physical[[49,50]] < 0.1), "Genome violates the new cross-projection limit"
print("Input:", GENOME_PATH.resolve() if GENOME_PATH.exists() else "generic configuration (untrained)")
print("Physical fields:", len(genome.names), "Independent coordinates:", genome.size)
print("Mirror constraints:", genome.controller.mirror_symmetry)
print("Mechanics [kg, m, viscosity, m/s^2]:", genome.controller.mechanics)
print("Command representation:", genome.kind)


## Inspect the physical parameters

Muscle and neural times are in milliseconds; geometry is in metres. Command coefficients use the ascending powers of desired angle divided by 45°. Amplitudes described as model units are not a calibrated physiological measurement.

In [ ]:
values = np.r_[physical, command.ravel()]
parameter_table = "| Parameter | Physical value |\n|---|---:|\n"
parameter_table += "\n".join(f"| `{name}` | {value:.8g} |" for name, value in zip(genome.names, values))
display(Markdown(parameter_table))


## Simulate and inspect fitness

All hidden neural and muscle states reset once at the start. The first call compiles Numba functions. Strict success requires surviving the episode and entering the joint angle/speed tolerance by the prescribed deadline; a small angle RMSE alone is insufficient.

In [ ]:
result = replay_genome(genome, unit, target_deg=TARGET_DEG,
                       angle_deg=INITIAL_ANGLE_DEG, speed_rad_s=INITIAL_SPEED_RAD_S,
                       duration_s=DURATION_S, stride=RECORD_EVERY,
                       fitness_window_s=FITNESS_WINDOW_S)
print("Fitness window (s):", result["fitness_window_s"])
print(json.dumps(dict(zip(METRIC_COLUMNS, result["metrics"].tolist())), indent=2))
figure = plot_replay(result)
plt.show()


## Activity of both motoneurons

Counts use every 0.1 ms spike sample. Complete late windows are split into common 1 s bins. This tests whether both neurons fired in a bin, not exact spike synchrony. An incomplete episode has unavailable late-window statistics instead of artificial zeros. The diagnostic never changes fitness or selection.

In [ ]:
if RECORD_EVERY != 1:
    raise ValueError("Set RECORD_EVERY=1 and rerun the simulation to count all spikes")
activity = measure_activity(result)
print(json.dumps(activity, indent=2))
print("Output P [muscle, neuron]:")
print(physical[48:52].reshape(2,2))


## Resolve the last 50 ms of neuronal activity

Use `RECORD_EVERY = 1` for this view; coarser recording may miss spike peaks.

In [ ]:
trace = result["trace"]
window = trace[:, 0] >= max(0, trace[-1, 0] - 0.05)
fig, ax = plt.subplots(figsize=(10, 3), layout="constrained")
ax.plot(trace[window, 0], trace[window, 5], label="Neuron 0")
ax.plot(trace[window, 0], trace[window, 6], label="Neuron 1")
ax.set(xlabel="Time (s)", ylabel="Membrane potential (mV)")
ax.legend(); ax.grid(alpha=0.2)
plt.show()


## Optional: start a new constrained training run

Training is disabled. Enabling the last cell starts fresh training from the current JSON configuration, not from the loaded learned controller. Use a new output folder. Full sweep commands are in the README.

In [ ]:
RUN_TRAINING = False
if RUN_TRAINING:
    import subprocess
    subprocess.run([sys.executable, str(ROOT / "run.py"), "train", EXPERIMENT.name[0],
                    "--profile", "fast", "--output", str(ROOT / "results" / f"{EXPERIMENT.name}_constrained_notebook")],
                   cwd=ROOT, check=True)
else:
    print("Training disabled.")
